In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql.functions import col

In [0]:
# Define table names using environment variables
silver_constructors_table = f"{catalog_name}.{silver_schema}.constructors"
gold_ref_nationality_region_table = f"{catalog_name}.{gold_schema}.ref_nationality_region"
gold_dim_constructors_table = f"{catalog_name}.{gold_schema}.dim_constructors"

# Read silver constructors table
constructors_df = spark.table(silver_constructors_table)

# Read gold ref_nationality_region table
ref_nationality_region_df = spark.table(gold_ref_nationality_region_table)

In [0]:
# Join constructors with ref_nationality_region on nationality
dim_constructors_df = (
    constructors_df.alias("constructor")
    .join(
        ref_nationality_region_df.alias("ref_nationality_region"),
        col("constructor.nationality") == col("ref_nationality_region.nationality"),
        "left"
    )
    # Select only required columns
    .select(
        col("constructor.constructor_id"),
        col("constructor.constructor_name"),
        col("constructor.nationality"),
        col("ref_nationality_region.region").alias("nationality_region")
    )
)

In [0]:
# Write transformed data to gold layer
(
    dim_constructors_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(gold_dim_constructors_table)
)

print(f"✓ Constructors dimension successfully written to {gold_dim_constructors_table}")

In [0]:
# Preview the constructors dimension table
#spark.table(gold_dim_constructors_table).limit(10).display()